# diag colab_11 — where did v2's fit actually land? A LoRA ablation ladder

`colab_11` v2 added `dense` to the LoRA target list and reached a slightly better loss than the
attention-only v1 pilot, while detector #1's embedding drift did **not** move (0.0057 → 0.0051).
Loss-fit and cosine-drift decoupled. This notebook asks where v2's extra fit physically went, and
whether detector #1 was positioned to see it.

**The geometry that motivates this.** `Geneformer-V2-104M` is a 12-layer BERT (hidden 768,
intermediate 3072). Embeddings are extracted with `emb_layer=-1`, which in Geneformer's convention
means the **2nd-to-last** layer — the output of layer index 10. Everything after that point is
invisible to detector #1: the whole of encoder layer 11, plus the MLM head. `dense` matched **37**
modules — 36 encoder projections and one head projection, `cls.predictions.transform.dense` — so
v2's added capacity is overwhelmingly encoder-side, and only a thin slice of it sits in the blind
spot:

| region | LoRA params | share of v2's trainable |
|---|---|---|
| MLM head (`cls.predictions.transform.dense`) | 12,288 | 0.9% |
| encoder layer 11 (qkv + all dense) | 110,592 | 8.3% |
| **both — detector #1's blind spot** | **122,880** | **9.2%** |
| v2 total trainable | 1,339,392 | 100% |

So detector #1 sees roughly 91% of the adapted capacity. The head-absorption hypothesis — that v2's
loss gain landed in the MLM head, downstream of the measurement — requires 0.9% of the parameters to
carry essentially all of the gain. Possible, since the head sits at the output where per-parameter
leverage on the logits is greatest, but it is a strong claim. This notebook tests it directly instead
of arguing about it.

**Method.** No retraining. Both LoRA adapters are on Drive; each variant is re-merged from the frozen
base checkpoint, with selected LoRA deltas zeroed, and evaluated on one **frozen masked validation
set** — the same masked tokens for every variant, so the only thing varying is model weights.

**The ladder.**

| rung | variant | question |
|---|---|---|
| 0 | base / v1 / v2 intact | is there a gain to attribute at all? |
| A | v2 − head | head absorption proper |
| B | v2 − layer 11 | last-encoder-layer absorption |
| C | v2 − both | how much of v2's fit lives in detector #1's blind spot |

**Two limits, stated up front.**

1. *Ablation is not decomposition.* The encoder trained **with** the head delta present. Zeroing it
   puts the model slightly off-distribution, so loss can land above v1's rather than reverting to it.
   Each rung yields an **upper bound** on its region's contribution, not an additive share, and the
   rungs need not sum to C.
2. *This cannot tell us the gap is real.* colab_11's validation curve has v2 lower at all 8
   checkpoints, widening from 0.0031 to 0.0049. That is weaker evidence than it looks: the 8
   checkpoints are 8 samples of the **same two trajectories**, near-perfectly autocorrelated, so
   "v2 wins 8/8" is close to one observation, not eight. v1 and v2 are N=1 each — they share
   `SEED=0`, but adding `dense` changes the trainable-parameter set, so LoRA init and dropout draws
   diverge and the runs are different trajectories. Separating "`dense` helps" from "this seed's
   trajectory" needs retraining, which is the cost this arc exists to avoid. §7b bounds only
   **measurement** noise — but that is not nothing here: v1 and v2 consumed different training RNG,
   so their eval masks likely differed too, and if the mask-draw spread turns out to be of order
   0.005 then the reference curve itself was never readable and there is no decoupling to explain.

Everything is measured on **val**, never test. Val is currently used only for CPT loss, so it is free
for this; choosing a representation or an interpretation on test and then running the contract's evals
on test would be selection on the evaluation surface.

## 1 — Setup

### 1a — Mount Drive, clone the repo, install the Geneformer environment

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
assert os.path.exists(DRIVE_ROOT), f"Drive root not found: {DRIVE_ROOT}"

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

# Lean Geneformer-only stack: rides Colab's native numpy-2 base (see NUMPY POLICY in
# requirements_geneformer.txt); scGPT is NOT installed here.
!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .

# Colab preinstalls torchao 0.10.0, unused here (no quantization, plain LoRA). peft's
# get_peft_model dispatches every target Linear through dispatch_torchao(), which calls
# is_torchao_available() -- if torchao is merely *importable*, that function hard-raises
# ImportError below its 0.16.0 floor rather than returning False. Uninstalling (not upgrading) is
# the minimal fix. Same treatment as colab_11, which this notebook must stay binary-comparable to.
!pip uninstall -y torchao

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True).stdout.strip()
print("Geneformer commit:", GENEFORMER_COMMIT)

### 1b — Run configuration

One `SMOKE` toggle drives everything. `SMOKE = True` subsamples the validation split to a few hundred
cells and collapses the noise-floor draws to one, so the whole notebook can be walked end-to-end in a
few minutes to prove the plumbing before an A100 hour is spent on it. Every output path is *derived*
from the toggle and carries a `_SMOKE` suffix, so a smoke run can waste time but can never overwrite a
real result. The split check in §3b runs **before** any subsampling, so it is never weakened by it.

In [ ]:
SMOKE = False            # live toggle -- committed False; flip in the Colab tab for a dry run

SEED           = 0       # matches colab_11's training seed
MLM_PROB       = 0.15    # MUST match CPT -- Geneformer's pretraining default
EVAL_BATCH     = 32      # forward-only; training used per-device 8 + grad-accum 4
N_MASK_SEEDS   = 4       # §7b: independent mask draws used to bound measurement noise
SMOKE_N_CELLS  = 512

# Reference numbers from colab_11 -- the run this notebook interrogates.
REF_TRAIN_LOSS = {"v1": 2.0971, "v2": 2.0921}

# colab_11's step-matched validation curve. v2 is lower at all 8 checkpoints and the gap widens
# monotonically (0.0031 -> 0.0049). Two cautions on reading that as a real effect: (a) these are 8
# samples of the SAME two trajectories, not 8 independent draws -- consecutive checkpoints are
# near-perfectly autocorrelated, so "v2 wins 8/8" carries almost no evidential weight; (b) v1 and
# v2 consumed different amounts of RNG during training (different param counts -> different dropout
# draws), so their eval masks almost certainly differed, and part of this gap may be mask noise.
# §7b measures the mask-draw spread and therefore calibrates whether this curve was ever readable.
REF_VAL_LOSS = {
    250:  {"v1": 2.093421, "v2": 2.090349},
    500:  {"v1": 2.091376, "v2": 2.088032},
    750:  {"v1": 2.089601, "v2": 2.085899},
    1000: {"v1": 2.089767, "v2": 2.086088},
    1250: {"v1": 2.088168, "v2": 2.084011},
    1500: {"v1": 2.087294, "v2": 2.082720},
    1750: {"v1": 2.086892, "v2": 2.082163},
    2000: {"v1": 2.086208, "v2": 2.081286},
}
REF_VAL_FINAL = REF_VAL_LOSS[2000]   # §7a reproduction target

RUN_TAG = "head_ablation" + ("_SMOKE" if SMOKE else "")

GF_DIR       = os.path.join(DRIVE_ROOT, "geneformer")
V1_ADAPTER   = os.path.join(GF_DIR, "cpt_aggregated_seed0_adapter")       # v1: query,key,value
V2_ADAPTER   = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")    # v2: + dense
MODEL_DIR    = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")

for p in (V1_ADAPTER, V2_ADAPTER, MODEL_DIR):
    assert os.path.exists(p), f"missing required input: {p}"

RESULTS_PATH = os.path.join(GF_DIR, f"cpt_{RUN_TAG}_results.json")
FREEZE_PATH  = os.path.join(GF_DIR, f"pip_freeze_{RUN_TAG}.txt")
ENV_JSON_PATH = os.path.join(GF_DIR, f"env_{RUN_TAG}.json")
AUDIT_KEY    = f"geneformer_cpt_{RUN_TAG}"

if SMOKE:
    N_MASK_SEEDS = 1
    print("*** SMOKE RUN *** val subsampled to", SMOKE_N_CELLS,
          "cells; 1 mask draw; writes ->", os.path.basename(RESULTS_PATH))
    print("*** results are plumbing evidence ONLY -- not a scientific readout ***")
else:
    print("LIVE RUN -> full val split,", N_MASK_SEEDS, "mask draws")

from datetime import date
TODAY = date.today().isoformat()
print("run tag:", RUN_TAG, "| date:", TODAY)

## 2 — Environment capture

### 2a — pip freeze + env JSON

The exact stack this measurement ran on. `pip_freeze.txt` is the one genuinely unrecoverable
artifact of a Colab session, so it is written to Drive before any compute.

In [ ]:
import json, platform, subprocess, sys, torch

os.makedirs(GF_DIR, exist_ok=True)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True, check=True).stdout
with open(FREEZE_PATH, "w") as f:
    f.write(freeze)

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
ENV = {
    "date": TODAY, "run_tag": RUN_TAG, "smoke": SMOKE,
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": gpu,
    "geneformer_commit": GENEFORMER_COMMIT,
}
with open(ENV_JSON_PATH, "w") as f:
    json.dump(ENV, f, indent=2)

print("pip freeze ->", FREEZE_PATH, f"({len(freeze.splitlines())} pkgs)")
print("env        ->", ENV_JSON_PATH)
print("GPU:", gpu, "| torch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU -- this notebook needs one (Runtime > Change runtime type)"

DEVICE = "cuda"

## 3 — Substrate + frozen split

### 3a — Load the glia substrate, guard raw counts

The same cell-for-cell substrate every FM notebook loads: 142,588 glia (54,805 microglia / 87,783
astrocytes), concatenated in the same order with the same `cell_index` key, so tokenization here is
bit-identical to colab_11's.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()

# concat order and index_unique MUST match colab_11 -- cell_index is assigned positionally and is
# the alignment key shared with the zero-shot baseline and every CPT embedding.
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("\ncombined glia:", glia.shape)

assert glia.n_obs == 142588, f"substrate is {glia.n_obs} cells, expected 142588 -- upstream changed"

# raw-counts guard -- Geneformer rank-encodes RAW counts; log-normalized input is silently wrong.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))

### 3b — Apply the frozen donor split and verify it against the reference

The standing split rule: this notebook **loads** `outputs/donor_split.json` and re-derives nothing.
A mismatch against the committed reference is a hard stop, not a licence to mint a new split — a
silently different val set would make every loss here incomparable to colab_11's.

In [ ]:
SPLIT_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_PATH), f"missing committed split {SPLIT_PATH}"
with open(SPLIT_PATH) as f:
    split_meta = json.load(f)

glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_meta["donor_split"])
unmapped = int(glia.obs["split"].isna().sum())
assert unmapped == 0, f"{unmapped} cells whose donor is absent from the frozen split"

REF = {"seed": 32, "margin": 10,
       "donors": {"train": 101, "val": 22, "test": 22},
       "cells":  {"train": 94963, "val": 23824, "test": 23801}}

obs_donors = {k: int(glia.obs.loc[glia.obs["split"] == k, "donor_id"].nunique())
              for k in ("train", "val", "test")}
obs_cells  = {k: int((glia.obs["split"] == k).sum()) for k in ("train", "val", "test")}

print("seed:", split_meta["seed"], "| test worst-case margin:", split_meta["test_worst_case_margin"])
print("donors:", obs_donors)
print("cells: ", obs_cells)

test_studies = (glia.obs.loc[glia.obs["split"] == "test", "study_id"]
                .value_counts(normalize=True).round(3).to_dict())
print("test study balance:", test_studies)

assert split_meta["seed"] == REF["seed"], f"split seed {split_meta['seed']} != reference {REF['seed']}"
assert split_meta["test_worst_case_margin"] == REF["margin"], "split margin != reference"
assert obs_donors == REF["donors"], f"donor counts {obs_donors} != reference {REF['donors']}"
assert obs_cells == REF["cells"], f"cell counts {obs_cells} != reference {REF['cells']}"
print("\nsplit verified against the frozen reference -- OK")

### 3c — Subset to the validation split

Tokenization is provably **cell-local**: the normalization factor comes from Geneformer's shipped
`gene_median_dict`, and the only per-cell quantity is that cell's own `n_counts` (verified against
`tokenizer.py` at the pinned commit — see `docs/ASSUMPTIONS.md`). Nothing depends on which other
cells are present, so tokenizing the val split alone yields tokens bit-identical to tokenizing all
142,588 and subsetting — at a sixth of the cost.

Any `SMOKE` subsample happens **here**, after the §3b verification has already passed on the full
object.

In [ ]:
val = glia[glia.obs["split"] == "val"].copy()
del glia; gc.collect()
print("val split:", val.shape)
print("  lineage:", val.obs["lineage"].value_counts().to_dict())
print("  donors: ", val.obs["donor_id"].nunique())

if SMOKE:
    rng = np.random.default_rng(SEED)
    take = rng.choice(val.n_obs, size=min(SMOKE_N_CELLS, val.n_obs), replace=False)
    val = val[np.sort(take)].copy()
    print(f"*** SMOKE: val subsampled -> {val.n_obs} cells ***")

# Geneformer's tokenizer reads n_counts off obs; compute over the full panel, per cell.
val.obs["n_counts"] = np.asarray(val.X.sum(axis=1)).ravel()
assert (val.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"
print("n_counts median:", float(np.median(val.obs["n_counts"])))

## 4 — Tokenize the validation split

### 4a — Map gene symbols to Ensembl IDs

Same mapping colab_09 and colab_11 used. The APOE hard-fail gate is re-asserted rather than assumed:
if the vocabulary ever shifted under us, this must stop here rather than produce a quietly different
tokenization.

In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
assert "<mask>" in token_dict and "<pad>" in token_dict, "token dictionary missing <mask>/<pad>"

val.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in val.var_names]
in_vocab = val.var["ensembl_id"].map(lambda e: (e in token_dict) if e is not None else False)
n_vocab = int(in_vocab.sum())
print(f"gene panel: {val.n_vars} | in Geneformer vocab: {n_vocab} ({n_vocab/val.n_vars:.1%})")

# Must reproduce colab_09's audited 63.33% -- a drift here means a different vocabulary.
assert abs(n_vocab / val.n_vars - 0.6333) < 0.005, (
    f"vocab coverage {n_vocab/val.n_vars:.4f} departs from colab_09's audited 0.6333")

apoe_e = symbol_to_ensembl.get("APOE")
assert apoe_e is not None and apoe_e in token_dict, "APOE not tokenizable -- pre-registered hard fail"
print("APOE in vocab:", apoe_e, "-- gate passed")

### 4b — Tokenize

Defaults are the ones colab_09 and colab_11 ran on, and are V2-shaped: `special_token=True`,
`model_input_size=4096`, `model_version="V2"` (verified — `docs/ASSUMPTIONS.md`). So `<cls>` is
prepended and `<eos>` appended to every cell, exactly as during CPT.

The `sum_ensembl_ids` monkeypatch is carried over verbatim from colab_11 §4b and is load-bearing:
without it, current pandas raises on the tokenizer's positional-indexing assumption.

In [ ]:
from geneformer import TranscriptomeTokenizer
import geneformer.tokenizer as _gftok

TOK_IN_DIR  = "/content/gf_token_in"
TOK_OUT_DIR = "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True)
os.makedirs(TOK_OUT_DIR, exist_ok=True)
val.write_h5ad(os.path.join(TOK_IN_DIR, "val.h5ad"))

CUSTOM_ATTRS = {
    "cell_index":   "cell_index",
    "split":        "split",
    "lineage":      "lineage",
    "substate":     "substate",
    "apoe_carrier": "apoe_carrier",
    "study_id":     "study_id",
    "donor_id":     "donor_id",
}

# Upstream Geneformer bug (confirmed against the pinned commit's tokenizer.py): tokenize_anndata
# does `adata.var["ensembl_id_collapsed"][coding_miRNA_loc]`, where coding_miRNA_loc is an array of
# integer POSITIONS, but var.index at that point is gene symbols/Ensembl IDs, never integers. Older
# pandas silently fell back to positional indexing; Colab's current pandas raises
# `KeyError: None of [...] are in the [index]`. Reset the post-sum_ensembl_ids var index to a plain
# RangeIndex so positions coincide with labels again -- restores exactly the behaviour this commit's
# code assumes, without touching pandas globally.
_orig_sum_ensembl_ids = _gftok.sum_ensembl_ids

def _sum_ensembl_ids_rangeindex_patch(*args, **kwargs):
    result = _orig_sum_ensembl_ids(*args, **kwargs)
    result.var.index = pd.RangeIndex(result.n_vars)
    return result

_gftok.sum_ensembl_ids = _sum_ensembl_ids_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "val_ablation", file_format="h5ad")

from datasets import load_from_disk
val_ds = load_from_disk(os.path.join(TOK_OUT_DIR, "val_ablation.dataset"))
print("tokenized val:", val_ds)
assert len(val_ds) == val.n_obs, f"tokenizer returned {len(val_ds)} cells, expected {val.n_obs}"

lens = np.array(val_ds["length"])
print(f"token lengths: median {np.median(lens):.0f} | min {lens.min()} | max {lens.max()}")

## 5 — Freeze one masked validation set

### 5a — Materialise fixed masked batches

MLM loss is stochastic in the masking. Comparing variants under independently drawn masks would put
mask noise directly into differences we expect to be on the order of 0.005 nats. So the masking is
done **once**, with Geneformer's own collator, and the resulting tensors are reused verbatim for
every variant: identical batch composition, identical padding, identical masked positions, identical
random-token substitutions. The only thing that varies across the ladder is model weights.

Note a real caveat inherited from upstream: `GeneformerPreCollator` registers only `<mask>` and
`<pad>` as special tokens, so `<cls>` and `<eos>` are ordinary maskable tokens and get masked in
roughly 15% of cells. That is genuine Geneformer CPT behaviour, and reproducing it here is what makes
these losses comparable to colab_11's — but it is a caveat if `<cls>` later becomes the readout.

In [ ]:
import torch
from transformers import DataCollatorForLanguageModeling
from geneformer.pretrainer import GeneformerPreCollator

precollator = GeneformerPreCollator(token_dictionary=token_dict)
collator = DataCollatorForLanguageModeling(tokenizer=precollator, mlm=True, mlm_probability=MLM_PROB)

def build_masked_batches(ds, batch_size, mask_seed):
    """Materialise a fixed set of masked batches. Same seed -> byte-identical batches."""
    torch.manual_seed(mask_seed)
    batches = []
    for i in range(0, len(ds), batch_size):
        rows = ds[i : i + batch_size]["input_ids"]
        batch = collator([{"input_ids": r} for r in rows])
        batches.append({k: v.cpu() for k, v in batch.items()})
    return batches

MASK_SEEDS = [SEED + i for i in range(N_MASK_SEEDS)]
frozen_batches = build_masked_batches(val_ds, EVAL_BATCH, MASK_SEEDS[0])

n_masked = sum(int((b["labels"] != -100).sum()) for b in frozen_batches)
n_tokens = sum(int(b["attention_mask"].sum()) for b in frozen_batches)
print(f"frozen masked set: {len(frozen_batches)} batches | {n_tokens:,} real tokens "
      f"| {n_masked:,} masked ({n_masked/n_tokens:.3%})")
assert abs(n_masked / n_tokens - MLM_PROB) < 0.02, "masking rate departs from MLM_PROB"

# determinism check -- rebuilding with the same seed must reproduce the set exactly, or every
# cross-variant comparison below is resting on an assumption that does not hold.
_check = build_masked_batches(val_ds.select(range(min(len(val_ds), 2 * EVAL_BATCH))), EVAL_BATCH, MASK_SEEDS[0])
for j, b in enumerate(_check):
    assert torch.equal(b["input_ids"], frozen_batches[j]["input_ids"]), \
        "masking is NOT reproducible under a fixed seed -- the ladder would be measuring mask noise"
print("masking determinism verified under fixed seed -- OK")
del _check

## 6 — Build the model variants

### 6a — Ablation machinery: zero a LoRA delta by module name

A LoRA module contributes `scaling * (B @ A)` to its base weight. Zeroing `B` removes that
contribution exactly, so `merge_and_unload()` then folds in a delta of zero and the module reverts to
its frozen pretrained weight. Every variant is loaded fresh from the base checkpoint so no ablation
can leak into the next one.

In [ ]:
from peft import PeftModel
from transformers import BertForMaskedLM

def load_variant(adapter_dir, zero_matching=()):
    """Re-merge a variant from the frozen base. `zero_matching` = substrings of module names whose
    LoRA delta is zeroed before merging. Returns (merged_model, {module_name: lora_params_zeroed})."""
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    if adapter_dir is None:
        assert not zero_matching, "cannot ablate the base model -- it carries no adapter"
        return base, {}, 0

    model = PeftModel.from_pretrained(base, adapter_dir)

    lora_modules = [(n, m) for n, m in model.named_modules() if hasattr(m, "lora_B") and len(m.lora_B)]
    n_lora = len(lora_modules)

    zeroed = {}
    for name, mod in lora_modules:
        if not any(pat in name for pat in zero_matching):
            continue
        with torch.no_grad():
            for k in mod.lora_B:
                mod.lora_B[k].weight.zero_()
        zeroed[name] = sum(mod.lora_A[k].weight.numel() + mod.lora_B[k].weight.numel()
                           for k in mod.lora_A)

    if zero_matching:
        assert zeroed, f"no LoRA module matched {zero_matching} -- the module tree is not what we think"

    merged = model.merge_and_unload()
    return merged, zeroed, n_lora

print("ablation machinery ready")

### 6b — Enumerate variants and verify the ablated parameter counts

This is the cell that proves the ladder is ablating what it claims to. The expected LoRA parameter
counts are **derived from the checkpoint's own config** (`r * (d_in + d_out)` per module) and
asserted against what was actually zeroed. If Geneformer's module tree, `peft`'s leaf-name matching,
or the LoRA rank is not what this notebook assumes, it stops here rather than reporting a confidently
wrong attribution.

In [ ]:
from transformers import BertConfig

cfg = BertConfig.from_pretrained(MODEL_DIR)
H, I, L = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers
R = 8   # colab_11's LORA_R, both runs
print(f"checkpoint: {L} layers | hidden {H} | intermediate {I} | lora r={R}")
assert (L, H, I) == (12, 768, 3072), f"checkpoint geometry changed: {(L, H, I)}"

LAST_LAYER = L - 1                       # emb_layer=-1 extracts the output of layer L-2, so layer
                                         # L-1 is downstream of the measurement point
HEAD_MATCH = "cls.predictions.transform.dense"
LAST_MATCH = f".layer.{LAST_LAYER}."

# expected LoRA params, derived not hardcoded: r*(d_in+d_out) per adapted module
p = lambda d_in, d_out: R * (d_in + d_out)
EXP_HEAD = p(H, H)                                                   # head transform dense
EXP_LAST = 3 * p(H, H) + p(H, H) + p(H, I) + p(I, H)                 # qkv + attn-out + FFN up/down
EXP_BOTH = EXP_HEAD + EXP_LAST
print(f"expected LoRA params -- head {EXP_HEAD:,} | layer {LAST_LAYER} {EXP_LAST:,} | both {EXP_BOTH:,}")

VARIANTS = [
    ("base",                None,       ()),
    ("v1_intact",           V1_ADAPTER, ()),
    ("v2_intact",           V2_ADAPTER, ()),
    ("v2_minus_head",       V2_ADAPTER, (HEAD_MATCH,)),
    ("v2_minus_last_layer", V2_ADAPTER, (LAST_MATCH,)),
    ("v2_minus_both",       V2_ADAPTER, (HEAD_MATCH, LAST_MATCH)),
]

EXPECT_ZEROED = {"v2_minus_head": EXP_HEAD, "v2_minus_last_layer": EXP_LAST, "v2_minus_both": EXP_BOTH}
EXPECT_N_LORA = {"v1_intact": 3 * L, "v2_intact": 6 * L + 1,
                 "v2_minus_head": 6 * L + 1, "v2_minus_last_layer": 6 * L + 1,
                 "v2_minus_both": 6 * L + 1}

ABLATION_AUDIT = {}
for name, adapter, pats in VARIANTS:
    if adapter is None:
        continue
    _m, zeroed, n_lora = load_variant(adapter, pats)
    total = sum(zeroed.values())
    ABLATION_AUDIT[name] = {"n_lora_modules": n_lora, "n_modules_zeroed": len(zeroed),
                            "lora_params_zeroed": total, "modules": sorted(zeroed)}
    print(f"\n{name}: {n_lora} LoRA modules | zeroed {len(zeroed)} module(s) = {total:,} params")
    for mn in sorted(zeroed):
        print(f"    {mn}  ({zeroed[mn]:,})")
    assert n_lora == EXPECT_N_LORA[name], \
        f"{name}: {n_lora} LoRA modules, expected {EXPECT_N_LORA[name]} -- target matching changed"
    if name in EXPECT_ZEROED:
        assert total == EXPECT_ZEROED[name], \
            f"{name}: zeroed {total:,} params, expected {EXPECT_ZEROED[name]:,}"
    del _m; gc.collect(); torch.cuda.empty_cache()

print("\nablation targets verified against the checkpoint config -- OK")

## 7 — The ladder

### 7a — Evaluate every variant on the frozen masked set

Loss is accumulated as a **token-weighted** sum over masked positions (`reduction="sum"` divided by
the true masked-token count), not a mean of per-batch means — batches carry different numbers of
masked tokens, and a mean-of-means would silently reweight them.

Read rung 0 first, as a **reproduction check**. colab_11's own validation curve already tells us v2
ends at 2.081286 and v1 at 2.086208, so these recomputed numbers should land near those — offset only
by the mask draw, since colab_11's eval masks are not recoverable. A large departure means this
notebook's eval differs *structurally* from colab_11's (different val set, different masking rate,
different token handling), and nothing below it is trustworthy. Rungs A–C are only worth reading once
rung 0 has reproduced.

The arc's real gate is §7b, not rung 0: the v1→v2 gap is ~0.0049, and if the mask-draw spread is of
that order then colab_11's curve was never readable and there is no decoupling to explain.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def eval_masked_loss(model, batches):
    model.eval().to(DEVICE)
    lossf = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=-100)
    tot_loss, tot_tok = 0.0, 0
    for b in tqdm(batches, leave=False):
        ids  = b["input_ids"].to(DEVICE)
        att  = b["attention_mask"].to(DEVICE)
        lab  = b["labels"].to(DEVICE)
        logits = model(input_ids=ids, attention_mask=att).logits
        tot_loss += float(lossf(logits.view(-1, logits.size(-1)), lab.view(-1)))
        tot_tok  += int((lab != -100).sum())
    return tot_loss / tot_tok, tot_tok

LADDER = {}
for name, adapter, pats in VARIANTS:
    model, zeroed, _ = load_variant(adapter, pats)
    loss, ntok = eval_masked_loss(model, frozen_batches)
    LADDER[name] = {"val_loss": loss, "n_masked_tokens": ntok,
                    "lora_params_zeroed": sum(zeroed.values())}
    print(f"{name:22s} val_loss {loss:.5f}  ({ntok:,} masked tokens)")
    del model; gc.collect(); torch.cuda.empty_cache()

print("\n--- rung 0a: does this eval reproduce colab_11's validation curve? ---")
REPRO = {}
for arm in ("v1", "v2"):
    got, ref = LADDER[f"{arm}_intact"]["val_loss"], REF_VAL_FINAL[arm]
    REPRO[arm] = {"recomputed": got, "colab_11_step2000": ref, "delta": got - ref}
    print(f"  {arm}: recomputed {got:.5f}  vs colab_11 step-2000 {ref:.5f}  (delta {got-ref:+.5f})")
print("  a delta far larger than §7b's mask-draw spread means this eval differs STRUCTURALLY from")
print("  colab_11's -- stop and reconcile before reading any rung below.")

print("\n--- rung 0b: is there a gain to attribute? ---")
d_cpt = LADDER["base"]["val_loss"] - LADDER["v2_intact"]["val_loss"]
d_v2v1 = LADDER["v1_intact"]["val_loss"] - LADDER["v2_intact"]["val_loss"]
REF_GAP = REF_VAL_FINAL["v1"] - REF_VAL_FINAL["v2"]
print(f"base - v2      (what CPT bought at all)   : {d_cpt:+.5f}")
print(f"v1   - v2      (the gap under attribution): {d_v2v1:+.5f}")
print(f"  colab_11 step-2000 val gap (reference)  : {REF_GAP:+.5f}")
print(f"  colab_11 train_loss gap (context)       : {REF_TRAIN_LOSS['v1'] - REF_TRAIN_LOSS['v2']:+.5f}")

print("\n--- rungs A/B/C: upper bounds on each region's contribution ---")
for rung, name in [("A", "v2_minus_head"), ("B", "v2_minus_last_layer"), ("C", "v2_minus_both")]:
    cost = LADDER[name]["val_loss"] - LADDER["v2_intact"]["val_loss"]
    frac = (cost / d_v2v1) if abs(d_v2v1) > 1e-9 else float("nan")
    LADDER[name]["ablation_cost"] = cost
    LADDER[name]["frac_of_v2_v1_gap"] = frac
    print(f"  {rung}  {name:22s} cost {cost:+.5f}  = {frac:6.1%} of the v1->v2 gap")

### 7b — Measurement noise floor: v2 across independent mask draws

`v2_intact` re-evaluated under fresh mask draws, weights held fixed. The spread is what this
instrument cannot resolve. Any ablation cost in §7a smaller than this spread is unreadable.

This bounds **measurement** noise only. It does **not** bound training-seed variance — v1 and v2 are
single runs on diverging RNG trajectories, and separating those would need retraining. So a v1→v2 gap
that clears this floor is *measured* reliably; it is still not thereby shown to be reproducible.

In [ ]:
NOISE = {}
if N_MASK_SEEDS > 1:
    model, _, _ = load_variant(V2_ADAPTER, ())
    for ms in MASK_SEEDS:
        batches = frozen_batches if ms == MASK_SEEDS[0] else build_masked_batches(val_ds, EVAL_BATCH, ms)
        loss, _ = eval_masked_loss(model, batches)
        NOISE[ms] = loss
        print(f"mask seed {ms}: val_loss {loss:.5f}")
        if ms != MASK_SEEDS[0]:
            del batches; gc.collect()
    del model; gc.collect(); torch.cuda.empty_cache()

    vals = np.array(list(NOISE.values()))
    spread = float(vals.max() - vals.min())
    print(f"\nmeasurement noise floor: sd {vals.std(ddof=1):.5f} | range {spread:.5f}")
    print(f"v1->v2 gap = {d_v2v1:+.5f}  ->  {abs(d_v2v1)/spread:.1f}x the mask-draw range"
          if spread > 0 else "")
    NOISE_SUMMARY = {"seeds": MASK_SEEDS, "losses": {str(k): v for k, v in NOISE.items()},
                     "sd": float(vals.std(ddof=1)), "range": spread}
else:
    print("*** SMOKE: noise floor skipped (N_MASK_SEEDS=1) ***")
    NOISE_SUMMARY = {"skipped": "smoke run"}

## 8 — Save + handoff

### 8a — Write results, append the audit trace, print commit commands

A smoke run writes only to `_SMOKE`-suffixed paths and does **not** touch `audit_report.json`, so it
can never contaminate the audit trail.

In [ ]:
import shlex

RESULTS = {
    "status": "computed", "date": TODAY, "smoke": SMOKE, "run_tag": RUN_TAG,
    "question": "where did colab_11 v2's loss gain land, and was detector #1 positioned to see it?",
    "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
    "surface": "val", "n_val_cells": int(val.n_obs), "eval_batch": EVAL_BATCH, "mlm_prob": MLM_PROB,
    "emb_layer_convention": "emb_layer=-1 extracts layer L-2 output; layer L-1 + MLM head are downstream",
    "geometry": {"n_layers": L, "hidden": H, "intermediate": I, "lora_r": R,
                 "lora_params_head": EXP_HEAD, "lora_params_last_layer": EXP_LAST,
                 "lora_params_blind_spot": EXP_BOTH, "v2_trainable": 1339392},
    "ablation_audit": ABLATION_AUDIT,
    "ladder": LADDER,
    "noise_floor": NOISE_SUMMARY,
    "ref_train_loss": REF_TRAIN_LOSS,
    "ref_val_loss_curve": {str(k): v for k, v in REF_VAL_LOSS.items()},
    "reproduction_check": REPRO,
    "limits": [
        "ablation is not decomposition: the encoder trained with the head delta present, so each "
        "rung is an upper bound on its region's contribution and rungs need not sum to C",
        "does not bound training-seed variance: v1/v2 are N=1 on diverging RNG trajectories",
        "colab_11's 8-checkpoint val curve is 8 samples of the same 2 trajectories, not 8 "
        "independent draws -- 'v2 wins 8/8' carries almost no evidential weight",
        "v1/v2 consumed different training RNG, so their eval masks likely differed; part of the "
        "reference curve's 0.0031-0.0049 gap may be mask noise, which §7b bounds",
    ],
}
with open(RESULTS_PATH, "w") as f:
    json.dump(RESULTS, f, indent=2)
print("results ->", RESULTS_PATH)

if SMOKE:
    print("\n*** SMOKE RUN -- audit_report.json NOT touched, nothing to commit ***")
else:
    AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
    report[AUDIT_KEY] = RESULTS
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("audit trace appended ->", AUDIT_REPORT_PATH, f"[{AUDIT_KEY}]")

    rel = os.path.relpath(AUDIT_REPORT_PATH, REPO_PATH)
    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    print(f"  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add {shlex.quote(rel)}")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          "'diag: LoRA ablation ladder -- attribute colab_11 v2 fit vs detector #1 blind spot'")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

### Carried forward

- `cpt_head_ablation_results.json` on Drive + the `geneformer_cpt_head_ablation` block in
  `outputs/audit_report.json` — the ladder, the ablation audit, and the noise floor.
- Both LoRA adapters remain the only durable CPT artifacts; every variant here was re-merged from
  base, and no merged checkpoint is persisted.
- Whatever rung 0 says governs what the B2 arc does next: a v1→v2 gap at the noise floor retires the
  loss/drift decoupling as a phenomenon; a gap that clears it hands the attribution to rungs A–C, and
  rung C specifically to the question of whether detector #1's extraction point needs to move.